# Prerequisites
본 `ipynb` 은 `Python=3.12` 에서 작성하였습니다. Package dependency 를 해결하기 위해 아래 cell 을 실행해주세요.

## Install Python packages

In [ ]:
%pip -q install -U openai==2.36.0 "openai[realtime]" sounddevice==0.5.5 dotenv==0.9.9 azure-ai-voicelive==1.2.0 websocket-client==1.9.0 aiohttp==3.14.0 azure-search-documents==11.7.0b2

## Load environment variables from a .env file
secret 노출을 피하고 notebook 들간의 일관된 환경변수를 설정하기 위해 `dotenv` 을 이용한다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_OPENAI_URI = os.getenv("AZURE_OPENAI_URI")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
AZURE_OPENAI_REALTIME_DEPLOYMENT = os.getenv("AZURE_OPENAI_REALTIME_DEPLOYMENT")
AZURE_AI_SEARCH_ENDPOINT = os.getenv("AZURE_AI_SEARCH_ENDPOINT")
AZURE_AI_SEARCH_ADMIN_KEY = os.getenv("AZURE_AI_SEARCH_ADMIN_KEY")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
AZURE_AI_SEARCH_INDEX = "news-ko-index"
AZURE_SPEECH_LEXICON_URL = os.getenv("AZURE_SPEECH_LEXICON_URL")

# RAG pipeline using voice models

## Upload documents into AI Search
RAG 에 사용될 documents 들을 AI Search 에 업로드한다.

In [ ]:
# resources/news_ko.csv 를 Azure AI Search 인덱스(news-ko-index)에 업로드한다.
# 인덱스 스키마는 JSON 이 아니라 azure-search-documents SDK 모델(코드)로 정의한다.
import csv

from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import ResourceNotFoundError
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchProfile,
    VectorSearchAlgorithmMetric,
    SemanticSearch,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
)
from openai import OpenAI

EMBED_DIM = 3072        # text-embedding-3-large 출력 차원
EMBED_MAX_CHARS = 6000  # 임베딩 토큰 한도(8191) 초과 방지용 글자수 제한
CSV_PATH = "resources/news_ko.csv"

cred = AzureKeyCredential(AZURE_AI_SEARCH_ADMIN_KEY)

# --- 1) 인덱스 스키마를 코드로 정의 (id, summary, category, content, embedding3) ---
index = SearchIndex(
    name=AZURE_AI_SEARCH_INDEX,
    fields=[
        # 키 필드
        SimpleField(
            name="id", type=SearchFieldDataType.String,
            key=True, filterable=True, sortable=True,
        ),
        # 한국어 분석기로 검색 가능한 요약 (큰 텍스트라 facet 비활성)
        SearchableField(
            name="summary", type=SearchFieldDataType.String,
            analyzer_name="ko.lucene", facetable=False,
        ),
        # 분류 (필터/패싯용)
        SimpleField(
            name="category", type=SearchFieldDataType.String,
            filterable=True, facetable=True, sortable=True,
        ),
        # 한국어 분석기로 검색 가능한 본문 (큰 텍스트라 facet 비활성)
        SearchableField(
            name="content", type=SearchFieldDataType.String,
            analyzer_name="ko.lucene", facetable=False,
        ),
        # 벡터 필드 (조회/저장 안 함, HNSW 프로파일 연결)
        SearchField(
            name="embedding3",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            hidden=True,    # retrievable=False
            stored=False,   # 원본 벡터 미저장으로 용량 절약
            vector_search_dimensions=EMBED_DIM,
            vector_search_profile_name="embedding3-profile",
        ),
    ],
    # 벡터 검색 설정 (HNSW + 코사인)
    vector_search=VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="hnsw_config",
                parameters=HnswParameters(
                    metric=VectorSearchAlgorithmMetric.COSINE,
                    m=4, ef_construction=400, ef_search=500,
                ),
            ),
        ],
        profiles=[
            VectorSearchProfile(
                name="embedding3-profile",
                algorithm_configuration_name="hnsw_config",
            ),
        ],
    ),
    # 시맨틱 랭킹 설정
    semantic_search=SemanticSearch(
        configurations=[
            SemanticConfiguration(
                name="default",
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name="summary"),
                    content_fields=[SemanticField(field_name="content")],
                    keywords_fields=[SemanticField(field_name="category")],
                ),
            ),
        ],
    ),
)

# 코드의 스키마가 기준이 되도록 기존 인덱스가 있으면 지우고 새로 만든다
index_client = SearchIndexClient(endpoint=AZURE_AI_SEARCH_ENDPOINT, credential=cred)
try:
    index_client.delete_index(AZURE_AI_SEARCH_INDEX)
    print(f"🗑️  기존 인덱스 '{AZURE_AI_SEARCH_INDEX}' 삭제")
except ResourceNotFoundError:
    pass
index_client.create_index(index)
print(f"✅ 인덱스 '{AZURE_AI_SEARCH_INDEX}' 생성 완료")

# --- 2) CSV 로드 후 content 를 임베딩 (azure_ai_search 와 동일한 임베딩 모델 사용) ---
with open(CSV_PATH, encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))
print(f"📄 CSV {len(rows)}건 로드: {CSV_PATH}")

embed_client_sync = OpenAI(
    base_url=AZURE_OPENAI_URI.rstrip("/") + "/openai/v1",
    api_key=AZURE_OPENAI_API_KEY,
)
texts = [(r.get("content") or " ")[:EMBED_MAX_CHARS] for r in rows]
emb_resp = embed_client_sync.embeddings.create(
    model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT, input=texts,
)
vectors = [d.embedding for d in emb_resp.data]
print(f"🧠 임베딩 {len(vectors)}건 완료 (차원 {len(vectors[0])})")

# --- 3) 문서로 변환 후 업로드 ---
documents = [
    {
        "id": r["id"],
        "summary": r.get("summary", ""),
        "category": r.get("category", ""),
        "content": r.get("content", ""),
        "embedding3": vec,
    }
    for r, vec in zip(rows, vectors)
]
search_client = SearchClient(
    endpoint=AZURE_AI_SEARCH_ENDPOINT,
    index_name=AZURE_AI_SEARCH_INDEX,
    credential=cred,
)
result = search_client.upload_documents(documents=documents)
ok = sum(1 for x in result if x.succeeded)
print(f"⬆️  업로드 완료: {ok}/{len(documents)} 성공")


## Voice live

**Voice Live API 도 tool calling(function calling) 을 지원**한다. 그래서 위 gpt-realtime 셀에서
직접 WebSocket 으로 구현한 RAG 를, `azure-ai-voicelive` SDK 로 동일하게 만들 수 있다.
세션에 `FunctionTool` 로 `search` 도구를 등록하면, 모델이 사실·지식 질문에서 도구를 호출하고
`azure_ai_search` 가 Azure AI Search(`news-ko-index`)를 검색해 그 결과로 음성 답변을 생성한다.
또한 에코 제거·노이즈 억제가 서버에서 처리되므로 gpt-realtime 셀의 마이크 게이팅 같은 수동 처리가
필요 없고, 스피커 데모에서도 끼어들기가 된다(발화 감지 시 `response.cancel()` 로 재생과 서버 응답을
동시에 중단). 재생부와 끼어들기 처리는 Azure 공식 VoiceLive 샘플 패턴을 따른다. 아래 셀은 이 로직을 basic 노트북과 동일한 `run_voicelive(...)` 함수로 묶고, `tools`/`on_tool` 로 RAG 검색(Azure AI Search)을 연결한다.

### 동작 원리 (블록도) — Voice Live + tool calling

```text
   🎙️ 마이크 ─ input_audio_buffer.append ─▶
  ╔═ Voice Live API (WebSocket) ══════════════════════════════════
  ║   에코제거·노이즈억제 → Server VAD → gpt-realtime
  ║                                          │ 도구 필요 판단
  ╚══════════════════════════════════════════┼═══════════════════
        │ ⓐ 일반 음성 응답                    ▼ ⓑ RESPONSE_FUNCTION_CALL_ARGUMENTS_DONE
        │                            ┌─ azure_ai_search(query) ──────────
        │                            │  쿼리 임베딩 → 하이브리드+시맨틱
        │                            │  → Azure AI Search (news-ko-index)
        │                            └───┬───────────────────────────────
        │                                │ conversation.item.create(FunctionCallOutputItem)
        │                                ▼ + response.create
        ◀ RESPONSE_AUDIO_DELTA ── 🔊 스피커   (검색 결과를 근거로 음성 답변)
```

**아래 코드의 이벤트 / API 대응**

| 단계 | Voice Live API |
|------|----------------|
| <small>도구 등록 | <small>`RequestSession(tools=[FunctionTool(...)], tool_choice=ToolChoiceLiteral.AUTO)`</small> |
| <small>마이크 → 서버</small> | <small>`input_audio_buffer.append`</small> |
| <small>발화 감지</small> | <small>`ServerEventType.INPUT_AUDIO_BUFFER_SPEECH_STARTED`</small> |
| <small>끼어들기</small> | <small>`speaker.clear()` + `conn.response.cancel()`</small> |
| <small>도구 호출 인자 완성</small> | <small>`ServerEventType.RESPONSE_FUNCTION_CALL_ARGUMENTS_DONE`</small> |
| <small>검색 결과 회신</small> | <small>`conversation.item.create(FunctionCallOutputItem(...))` → `response.create()`</small> |
| <small>응답 오디오</small> | <small>`ServerEventType.RESPONSE_AUDIO_DELTA`</small> |
| <small>응답 오디오 종료</small> | <small>`ServerEventType.RESPONSE_DONE`</small> |


In [ ]:
import json
import asyncio
import base64
import threading
import array

import httpx
import sounddevice as sd
from openai import AsyncOpenAI
from azure.core.credentials import AzureKeyCredential
from azure.ai.voicelive.aio import connect
from azure.ai.voicelive.models import (
    RequestSession, Modality, InputAudioFormat, OutputAudioFormat,
    AzureSemanticVadMultilingual, ServerEventType, AzureStandardVoice, AudioInputTranscriptionOptions,
    AudioEchoCancellation, AudioNoiseReduction, FunctionTool, ToolChoiceLiteral,
    FunctionCallOutputItem,
)

# 재실행 시 남아 있는 PortAudio 상태 초기화 (-9986 예방)
def reset_portaudio():
    try:
        sd._terminate()
    except Exception as e:
        print(f"PortAudio reset error: {e}")
    sd._initialize()

# 스피커 — 페이드아웃 + 세대(generation) 차단
#  clear() 가 세대를 올려, 끊긴 응답의 늦게 도착한 오디오를 add() 가 전부 버린다(잔여음 누수 0)
class Speaker:
    def __init__(self, samplerate=24000, blocksize=1200, fade_sec=0.5):   # 1200 frames = 50ms
        self.buf, self.pos = bytearray(), 0
        self.lock = threading.Lock()
        self.gen = 0                              # 응답 세대
        self.fade = int(samplerate * fade_sec)    # 끼어들기 페이드아웃 길이(샘플)
        self.stream = sd.RawOutputStream(
            samplerate=samplerate, channels=1, dtype="int16",
            blocksize=blocksize, callback=self._cb)

    def _cb(self, outdata, frames, time, status):
        need = len(outdata)
        with self.lock:
            avail = len(self.buf) - self.pos
            n = min(need, avail)
            outdata[:n] = bytes(self.buf[self.pos:self.pos + n]); self.pos += n
            if n < need:
                outdata[n:] = b"\x00" * (need - n)   # 부족분은 무음
            if self.pos > 2_000_000:
                del self.buf[:self.pos]; self.pos = 0

    def add(self, pcm, gen):
        with self.lock:
            if gen != self.gen:        # 끊긴(옛 세대) 응답 오디오는 버림 → "다른 음성" 누수 차단
                return
            self.buf.extend(pcm)

    def clear(self):                   # 끼어들기: 세대 증가 + 페이드아웃
        with self.lock:
            self.gen += 1              # 이후 도착하는 옛 응답 delta 전부 무시
            cur = bytes(self.buf[self.pos:]); self.buf = bytearray(); self.pos = 0
            s = array.array("h"); s.frombytes(cur[:self.fade * 2]); m = len(s)
            for i in range(m):
                s[i] = int(s[i] * (m - i) / m)   # 선형 페이드아웃(즉시 끊으면 '탁' 클릭음)
            self.buf.extend(s.tobytes())

    def start(self):
        self.stream.start()
    def close(self):
        try: self.stream.stop()
        finally: self.stream.close()


def _build_voice(voice, lexicon_url=None):
    """보이스 자동 분기: '-'/':' 포함이면 Azure 뉴럴 보이스, 아니면 OpenAI 네이티브 보이스(문자열)."""
    if not isinstance(voice, str):
        return voice                       # 이미 구성된 voice 객체는 그대로
    if ("-" in voice) or (":" in voice):   # 예: ko-kr-SunHi:DragonHDLatestNeural
        kw = {"name": voice}
        if lexicon_url:
            kw["custom_lexicon_url"] = lexicon_url   # 발음 사전(Azure 보이스 전용)
        return AzureStandardVoice(**kw)
    return voice                           # 예: marin, alloy (OpenAI 보이스)


async def run_voicelive(
    model: str,
    voice,
    instructions: str,
    *,
    lexicon_url: str | None = None,        # 출력 TTS 발음 사전 (Azure 보이스 전용)
    phrase_list: list | None = None,       # 입력 STT 힌트 (cascaded/azure-speech 전용)
    transcription_model: str = "azure-speech",
    language: str = "ko",
    tools: list | None = None,             # RAG 등 function tool 목록
    on_tool=None,                          # async (name, args)->str : 도구 실행 콜백
    vad_threshold: float = 0.5,
    silence_ms: int = 500,
    fade_sec: float = 0.5,                 # 끼어들기 페이드아웃 길이(초)
    intro: str = "🎙️  말해보세요.\n",
):
    """Voice Live 음성 대화 공용 함수 (basic.ipynb 와 동일).
    - cascaded : model=gpt-5.4(chat) + voice=DragonHD(Azure)  → lexicon/phrase_list 사용 가능
    - S2S      : model=gpt-realtime  + voice=OpenAI 보이스(marin 등)
    서버 AEC·끼어들기(response.cancel)·세대 차단(끊긴 응답 잔여음 제거)·페이드아웃 공통 적용. tools/on_tool 로 RAG.
    """
    reset_portaudio()
    speaker = mic = sender = None
    state = {"active": False, "done": False}

    session_kwargs = dict(
        modalities=[Modality.TEXT, Modality.AUDIO],
        instructions=instructions,
        voice=_build_voice(voice, lexicon_url),
        input_audio_format=InputAudioFormat.PCM16,
        output_audio_format=OutputAudioFormat.PCM16,
        turn_detection=AzureSemanticVadMultilingual(threshold=vad_threshold, prefix_padding_ms=300,
                                 silence_duration_ms=silence_ms, remove_filler_words=True),
        input_audio_echo_cancellation=AudioEchoCancellation(),
        input_audio_noise_reduction=AudioNoiseReduction(type="azure_deep_noise_suppression"),
    )
    if phrase_list:
        session_kwargs["input_audio_transcription"] = AudioInputTranscriptionOptions(
            model=transcription_model, language=language, phrase_list=phrase_list)
    if tools:
        session_kwargs["tools"] = tools
        session_kwargs["tool_choice"] = ToolChoiceLiteral.AUTO

    try:
        speaker = Speaker(fade_sec=fade_sec)
        loop = asyncio.get_running_loop()
        mic_q: asyncio.Queue[bytes] = asyncio.Queue()

        def mic_cb(indata, frames, time, status):
            loop.call_soon_threadsafe(mic_q.put_nowait, bytes(indata))

        mic = sd.RawInputStream(samplerate=24000, channels=1, dtype="int16",
                                blocksize=1200, latency="high", callback=mic_cb)

        async with connect(endpoint=AZURE_OPENAI_URI,
                           credential=AzureKeyCredential(AZURE_OPENAI_API_KEY),
                           model=model) as conn:
            await conn.session.update(session=RequestSession(**session_kwargs))
            speaker.start(); mic.start()
            print(intro)

            async def pump_mic():
                while True:
                    chunk = await mic_q.get()
                    await conn.input_audio_buffer.append(audio=base64.b64encode(chunk).decode())
            sender = asyncio.create_task(pump_mic())

            cur_gen = 0
            async for event in conn:
                if event.type == ServerEventType.RESPONSE_CREATED:
                    state["active"], state["done"] = True, False
                    cur_gen = speaker.gen                 # 이 응답의 세대 고정
                elif event.type == ServerEventType.RESPONSE_AUDIO_DELTA:
                    speaker.add(event.delta, cur_gen)     # 세대와 함께 add
                elif event.type == ServerEventType.RESPONSE_AUDIO_TRANSCRIPT_DELTA:
                    print(event.delta, end="", flush=True)
                elif event.type == ServerEventType.CONVERSATION_ITEM_INPUT_AUDIO_TRANSCRIPTION_COMPLETED:
                    print(f"\n🧑 {event.transcript}")
                elif event.type == ServerEventType.INPUT_AUDIO_BUFFER_SPEECH_STARTED:
                    print("\n🎧 음성 인식 시작...")
                    speaker.clear()                       # gen++ → 옛 delta 자동 차단
                    if state["active"] and not state["done"]:
                        try:
                            await conn.response.cancel()
                        except Exception as e:
                            if "no active response" not in str(e).lower():
                                print(f"\n(취소 경고: {e})")
                elif event.type == ServerEventType.RESPONSE_FUNCTION_CALL_ARGUMENTS_DONE:
                    args = json.loads(event.arguments)
                    print(f"\n🔧 도구 호출: {event.name}({args})")
                    result = await on_tool(event.name, args) if on_tool else "도구가 없습니다."
                    await conn.conversation.item.create(
                        item=FunctionCallOutputItem(call_id=event.call_id, output=result))
                    await conn.response.create()
                elif event.type == ServerEventType.RESPONSE_DONE:
                    state["active"], state["done"] = False, True
                    print("\n✅ 응답 완료\n")
                elif event.type == ServerEventType.ERROR:
                    msg = getattr(event.error, "message", str(event.error))
                    if "no active response" not in msg.lower():
                        print("\n⚠️  ERROR:", msg)
    finally:
        if sender:
            sender.cancel()
        if mic:
            try: mic.stop(); mic.close()
            except Exception as e: print(f"마이크 종료 에러: {e}")
        if speaker:
            try: speaker.close()
            except Exception as e: print(f"스피커 종료 에러: {e}")


# ── RAG: 임베딩 클라이언트 + Azure AI Search 조회 + search 도구 ──
embed_client = AsyncOpenAI(
    base_url=AZURE_OPENAI_URI.rstrip("/") + "/openai/v1",
    api_key=AZURE_OPENAI_API_KEY,
)

async def azure_ai_search(query: str, k: int = 3) -> str:
    emb = await embed_client.embeddings.create(
        model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT, input=[query],
    )
    qvec = emb.data[0].embedding
    body = {
        "search": query,
        "searchFields": "summary,content",
        "vectorQueries": [
            {"kind": "vector", "vector": qvec, "fields": "embedding3", "k": k},
        ],
        "queryType": "semantic",
        "semanticConfiguration": "default",
        "select": "category,summary,content",
        "top": k,
    }
    url = (
        f"{AZURE_AI_SEARCH_ENDPOINT}/indexes/{AZURE_AI_SEARCH_INDEX}"
        f"/docs/search?api-version=2024-07-01"
    )
    async with httpx.AsyncClient(timeout=30) as http:
        resp = await http.post(url, json=body, headers={
            "api-key": AZURE_AI_SEARCH_ADMIN_KEY,
            "Content-Type": "application/json",
        })
        resp.raise_for_status()
        results = resp.json().get("value", [])
    if not results:
        return "관련 정보를 찾지 못했습니다."
    chunks = []
    for r in results:
        text = (r.get("content") or "").strip().replace("\n", " ")[:500]
        chunks.append(f"[{r.get('category')}] {text}")
    return "\n\n".join(chunks)

search_tool = FunctionTool(
    name="search",
    description=(
        "한국어 지식/뉴스 기사(인공지능·전기차·비트코인·손흥민·BTS·김치·제주도 등)가 "
        "색인된 Azure AI Search 에서 관련 정보를 검색합니다."
    ),
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string", "description": "검색어"}},
        "required": ["query"],
    },
)

### (Demo) Voice-Live api -> S2S(realtime) + RAG

In [ ]:
await run_voicelive(
    model="gpt-realtime-1.5",
    voice="marin",                # alloy / echo / shimmer / marin / cedar ...
    instructions=(
        "너는 삼성전자 Customer Service를 담당하는 음성 기반 AI 상담사야. 사용자와 실시간 음성 대화를 하는 환경에서 모든 응답은 4문장을 넘지 않게 말하고, 실제 사람이 말하듯 자연스럽고 간결하게 표현해"
        "사용자가 사실·지식·뉴스를 물으면 추측하지 말고, "
        "'잠깐만요'·'찾아볼게요' 같은 사전 멘트 없이 곧바로 search 도구를 호출해. 검색 결과가 오면 그 내용을 근거로 답해. "
        "답변은 화면이 아니라 소리로 전달되니, 불릿·목록·마크다운·괄호 나열을 쓰지 말고 사람이 말하듯 자연스러운 완결된 문장으로 "
        "한 번에 2~3문장으로 핵심만 짧게 말해줘. 날짜·숫자는 풀어 읽고(예: '1992년생'), 정보가 많으면 다 나열하지 말고 "
        "가장 중요한 것부터 말한 뒤 '더 자세히 알려드릴까요?'처럼 되물어 대화를 이어가. 검색 결과에 없는 내용이면 모른다고 솔직히 말해."
    ),
    tools=[search_tool],
    on_tool=lambda name, args: azure_ai_search(args["query"]),
    intro="\U0001f399️  말해보세요. 예: '손흥민에 대해 알려줘', '비트코인이 뭐야'.\n",
)

### (Demo) Voice-Live api -> CaseCade(gpt-5.4) + RAG

In [ ]:
await run_voicelive(
    model=AZURE_OPENAI_CHAT_DEPLOYMENT,
    voice="ko-kr-Sunhi:DragonHDLatestNeural",
    # voice="ko-kr-Hyunsu:DragonHDLatestNeural",
    # voice="en-us-Andrew:DragonHDLatestNeural",
    # voice="en-us-Ava3:DragonHDLatestNeural",
    lexicon_url=AZURE_SPEECH_LEXICON_URL,   # (참고용) 남겨두지만 핵심은 아래 instruction
    instructions=(
        "너는 삼성전자 Customer Service를 담당하는 음성 기반 AI 상담사야. 사용자와 실시간 음성 대화를 하는 환경에서 모든 응답은 4문장을 넘지 않게 말하고, 실제 사람이 말하듯 자연스럽고 간결하게 표현해"
        "사용자가 사실·지식·뉴스를 물으면 추측하지 말고, "
        "'잠깐만요'·'찾아볼게요' 같은 사전 멘트 없이 곧바로 search 도구를 호출해. 검색 결과가 오면 그 내용을 근거로 답해. "
        "답변은 화면이 아니라 소리로 전달되니, 불릿·목록·마크다운·괄호 나열을 쓰지 말고 사람이 말하듯 자연스러운 완결된 문장으로 "
        "한 번에 2~3문장으로 핵심만 짧게 말해줘. 날짜·숫자는 풀어 읽고(예: '1992년생'), 정보가 많으면 다 나열하지 말고 "
        "가장 중요한 것부터 말한 뒤 '더 자세히 알려드릴까요?'처럼 되물어 대화를 이어가. 검색 결과에 없는 내용이면 모른다고 솔직히 말해."
    ),
    tools=[search_tool],
    on_tool=lambda name, args: azure_ai_search(args["query"]),
)